In [9]:
import psutil
ram = psutil.virtual_memory()
print(f"RAM: {ram.used/1e9:.1f}GB used / {ram.total/1e9:.1f}GB total / {ram.available/1e9:.1f}GB free")

RAM: 7.7GB used / 8.2GB total / 0.6GB free


In [10]:
# ── MEMORY SAFE LOAD ─────────────────────────────
import torch
import gc

# Clear any previous model from memory
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
except:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Check available memory first
if torch.cuda.is_available():
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    free   = (torch.cuda.get_device_properties(0).total_memory 
              - torch.cuda.memory_allocated()) / 1e9
    print(f"GPU: {total:.1f}GB total | {free:.1f}GB free")

# Load on CPU first, then move to device
model = GPT()
state = torch.load(
    "LLM/model_weights.pt",
    map_location="cpu"          # ← load to CPU first, avoids OOM spike
)
model.load_state_dict(state)
del state                        # ← free the raw dict immediately
gc.collect()

model = model.to(DEVICE)
model.eval()
print("✅ Model loaded safely")

NameError: name 'GPT' is not defined

In [3]:
# Cell 1 — Install
!pip install bitsandbytes --quiet

In [11]:
# Cell 2 — Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import re, gc
from tokenizers import Tokenizer

In [12]:
# Cell 3 — Full GPT Model Class

BLOCK_SIZE = 256
VOCAB_SIZE  = 16000
N_EMBED     = 384
N_HEAD      = 6
N_LAYER     = 6
DROPOUT     = 0.2

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(N_EMBED, head_size, bias=False)
        self.query = nn.Linear(N_EMBED, head_size, bias=False)
        self.value = nn.Linear(N_EMBED, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        k   = self.key(x)
        q   = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads   = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj    = nn.Linear(head_size * num_heads, N_EMBED)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.GELU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffw  = FeedForward(n_embed)
        self.ln1  = nn.LayerNorm(n_embed)
        self.ln2  = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffw(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding    = nn.Embedding(VOCAB_SIZE, N_EMBED)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, N_EMBED)
        self.blocks             = nn.Sequential(*[Block(N_EMBED, N_HEAD) for _ in range(N_LAYER)])
        self.ln_final           = nn.LayerNorm(N_EMBED)
        self.lm_head            = nn.Linear(N_EMBED, VOCAB_SIZE, bias=True)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T))  # CPU only
        x       = tok_emb + pos_emb
        x       = self.blocks(x)
        x       = self.ln_final(x)
        logits  = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

print("✅ Model class ready")

✅ Model class ready


In [13]:
# Cell 4 — Load + Quantize
import gc

gc.collect()

# ── Correct path ──
model = GPT()
state = torch.load(r"C:/Users/manis/LLM/model_weights.pt", map_location="cpu")
model.load_state_dict(state)
del state
gc.collect()

# ── Quantize to int8 ──
model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)
model.eval()

import psutil
ram = psutil.virtual_memory()
print(f"✅ Quantized | RAM free: {ram.available/1e9:.2f}GB")

C:\Users\manis\AppData\Local\Temp\ipykernel_59092\201198001.py:14: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model = torch.quantization.quantize_dynamic(


✅ Quantized | RAM free: 0.68GB


In [14]:
# Cell 5 — Load tokenizer
tokenizer = Tokenizer.from_file(r"C:/Users/manis/LLM/medical_tokenizer.json")
print("Tokenizer OK:", tokenizer.get_vocab_size())

Tokenizer OK: 16000


In [15]:
# Cell 6 — Test (run this now)
from tokenizers import Tokenizer
import re

STOP_PHRASES = [
    "### Instruction", "### Question", "Q :", "Which of",
    "A patient", "The correct answer", "Ans .", "Ref :", "# ", "\n\n",
]

def ask(question, temperature=0.5, max_new_tokens=60, top_k=30):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    ids    = tokenizer.encode(prompt).ids
    idx    = torch.tensor([ids])
    token_counts = {}

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -256:])
            logits     = logits[:, -1, :].clone()

            for tid, cnt in token_counts.items():
                logits[:, tid] /= (1.8 ** min(cnt, 4))

            logits /= temperature
            top_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_vals[:, [-1]]] = float("-inf")

            next_tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
            tid = next_tok.item()
            token_counts[tid] = token_counts.get(tid, 0) + 1
            if token_counts[tid] >= 4:
                break
            idx = torch.cat([idx, next_tok], dim=1)

    raw = tokenizer.decode(idx[0][len(ids):].tolist())
    for stop in STOP_PHRASES:
        if stop in raw:
            raw = raw[:raw.index(stop)]

    sentences = re.split(r'(?<=[.!?])\s+', raw.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 15]
    result    = sentences[0] if sentences else raw.strip()
    result    = re.sub(r'\s([.,;:!?])', r'\1', result)
    return re.sub(r'\s+', ' ', result).strip()

# ── One question at a time to save memory ──
print("Q: What is Metformin used for?")
print("A:", ask("What is Metformin used for?"))

Q: What is Metformin used for?
A: 


In [16]:
# Debug — see raw output before cleaning
def ask_raw(question, temperature=0.5, max_new_tokens=60, top_k=30):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    ids    = tokenizer.encode(prompt).ids
    idx    = torch.tensor([ids])
    token_counts = {}

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -256:])
            logits     = logits[:, -1, :].clone()

            for tid, cnt in token_counts.items():
                logits[:, tid] /= (1.8 ** min(cnt, 4))

            logits /= temperature
            top_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_vals[:, [-1]]] = float("-inf")

            next_tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
            tid = next_tok.item()
            token_counts[tid] = token_counts.get(tid, 0) + 1
            if token_counts[tid] >= 4:
                break
            idx = torch.cat([idx, next_tok], dim=1)

    raw = tokenizer.decode(idx[0][len(ids):].tolist())
    print("RAW OUTPUT:")
    print(repr(raw))   # shows exact characters including \n
    print("─" * 50)
    print("VISIBLE:")
    print(raw)

ask_raw("What is Metformin used for?")

RAW OUTPUT:
'# Nicot inic acid : - Keto glutarate dehydrogenase ( D ) S ar ine . A patient presented with acute onset breathlessness and a history of fever , cough productive for 2 days duration & respiratory distress syndrome is most likely to have which ? The correct answer is C : Fever'
──────────────────────────────────────────────────
VISIBLE:
# Nicot inic acid : - Keto glutarate dehydrogenase ( D ) S ar ine . A patient presented with acute onset breathlessness and a history of fever , cough productive for 2 days duration & respiratory distress syndrome is most likely to have which ? The correct answer is C : Fever


In [17]:
import re

def ask(question, temperature=0.55, max_new_tokens=80, top_k=35):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    ids    = tokenizer.encode(prompt).ids
    idx    = torch.tensor([ids])
    token_counts = {}

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -256:])
            logits     = logits[:, -1, :].clone()

            for tid, cnt in token_counts.items():
                logits[:, tid] /= (1.8 ** min(cnt, 4))

            logits /= temperature
            top_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_vals[:, [-1]]] = float("-inf")

            next_tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
            tid = next_tok.item()
            token_counts[tid] = token_counts.get(tid, 0) + 1
            if token_counts[tid] >= 4:
                break
            idx = torch.cat([idx, next_tok], dim=1)

    raw = tokenizer.decode(idx[0][len(ids):].tolist())

    # ── Extract best sentence ────────────────────
    # Split into all sentences
    sentences = re.split(r'(?<=[.!?])\s+', raw.strip())

    # Score each sentence — higher = more like an answer
    def score(s):
        s_lower = s.lower()
        good = ["used for", "treatment", "caused by", "mechanism",
                "diabetes", "insulin", "blood", "glucose", "pressure",
                "reduces", "inhibit", "increases", "decreases", "acts"]
        bad  = ["correct answer", "patient presented", "which of",
                "ans ", "ref :", "a patient", "most likely", "following"]
        score = sum(2 for w in good if w in s_lower)
        score -= sum(3 for w in bad  if w in s_lower)
        score += min(len(s) / 20, 3)   # prefer longer sentences
        return score

    # Filter and rank
    candidates = [s.strip() for s in sentences if len(s.strip()) > 20]
    if not candidates:
        return "Model could not generate a clean answer."

    best = max(candidates, key=score)

    # Clean up spacing
    best = re.sub(r'\s([.,;:!?])', r'\1', best)
    best = re.sub(r'\s+', ' ', best).strip()
    return best


# ── Test ────────────────────────────────────────
questions = [
    "What is Metformin used for?",
    "What are the symptoms of diabetes?",
    "What causes hypertension?",
    "What is the treatment for pulmonary embolism?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("─" * 50)

Q: What is Metformin used for?
A: # Insulin: Glucagon and norepinephrine A 40 year old woman presents to her physician complaining of palpitations.
──────────────────────────────────────────────────
Q: What are the symptoms of diabetes?
A: Hepatic duct arises from intestine to bile into biliary tract ( hepatic venous ) by - it forms intestinal ester ified gut through chylomicrons with cholesterol transport via LDL receptors; VLDL secretion.# It causes glycogen esis.> A large amount for glucose transpoer also provides
──────────────────────────────────────────────────
Q: What causes hypertension?
A: Hyper lipidemia: seen with hypertension, hyperlipidemia and diabetes mellitus.
──────────────────────────────────────────────────
Q: What is the treatment for pulmonary embolism?
A: The initial treatment is best treated with pulmonary embolism ( MI ) or intravenous administration of diuretics are the drug for a cardiac catheter.# Ventricular fibrillation due to an opening in atrial septal de

In [18]:
# These match your ACTUAL training data format
# Your model learned MedMCQA exam text, so ask in that style

from tokenizers import Tokenizer
import torch, torch.nn.functional as F, re

tokenizer = Tokenizer.from_file(r"C:/Users/manis/LLM/medical_tokenizer.json")

def ask(prompt, temperature=0.6, max_new_tokens=80, top_k=40):
    ids  = tokenizer.encode(prompt).ids
    idx  = torch.tensor([ids])
    token_counts = {}

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, _ = model(idx[:, -256:])
            logits = logits[:, -1, :].clone()
            for tid, cnt in token_counts.items():
                logits[:, tid] /= (1.6 ** min(cnt, 4))
            logits /= temperature
            top_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_vals[:, [-1]]] = float("-inf")
            next_tok = torch.multinomial(F.softmax(logits, dim=-1), 1)
            tid = next_tok.item()
            token_counts[tid] = token_counts.get(tid, 0) + 1
            if token_counts[tid] >= 4:
                break
            idx = torch.cat([idx, next_tok], dim=1)

    return tokenizer.decode(idx[0][len(ids):].tolist())

# ── Questions that MATCH your training data style ──────────────
tests = [
    # MedMCQA style (most of your data)
    "The mechanism of action of Metformin is",
    "Drug of choice for type 2 diabetes is",
    "Symptoms of pulmonary embolism include",
    "First line treatment for hypertension is",
    "Troponin is used as a marker for",
    "The most common cause of myocardial infarction is",
]

for prompt in tests:
    print(f"\nPROMPT: {prompt}")
    print(f"OUTPUT: {ask(prompt)}")
    print("─" * 55)


PROMPT: The mechanism of action of Metformin is
OUTPUT: In the breakdown of an enzyme involved in biosynthesis is also responsible for this pathway . The most common site among - The correct answer is A : Membranous wall . Ans ) ( b .) ): Ref Harrison 18th ed , p 18 24 * Most commonly fractured end one .* MC type of tympan o fracture can cause a cast or medial condyle .* Posterior dislocation *** Fracture femur # humerus
───────────────────────────────────────────────────────

PROMPT: Drug of choice for type 2 diabetes is
OUTPUT: Treatment of choice for hypertension is The correct answer is B : Anti - blockers . All the following are true about a patient with an increased risk factor except ? The correct answer was D : Increased incidence in blood pressure and decreased total volume increases levels increase in plasma level , Decreased ICP ( N MS ) has been mentioned to be associated with poor prognosis than those who
───────────────────────────────────────────────────────

PROMPT: Sy